# Detecção de Fraudes e Segmentação de Clientes em Cartões de Crédito (Pipeline KDD)

Este estudo aplica a metodologia **KDD (*Knowledge Discovery in Databases*)** sobre a base exclusiva `base04_cartao_alta_dimensao.csv`, composta por registros transacionais e comportamentais de clientes de uma operadora de cartões de crédito sem rótulos pré-definidos.

O fluxo de trabalho está organizado nas cinco etapas clássicas do KDD:
1. **Seleção e Diagnóstico dos Dados**: importação da base, inspeção da dimensionalidade, tipagem das variáveis, verificação de dados nulos e remoção de identificadores sem valor analítico.
2. **Análise Exploratória de Dados (AED)**: investigação da história comportamental contada pelas variáveis de limite, consumo, canais de pagamento, tipos de lojistas e indicadores de risco (saques no crédito, compras na madrugada, estornos e alterações cadastrais).
3. **Pré-processamento**: codificação de variáveis categóricas nominais via *One-Hot Encoding*, mapeamento da matriz de correlação com eliminação de atributos colineares ($|r| > 0,90$) e remoção de outliers univariados extremos ($4,0 \times \text{IQR}$).
4. **Transformação**: padronização com `RobustScaler` e redução de dimensionalidade linear via Análise de Componentes Principais (PCA), com nova inspeção gráfica pós-tratamento.
5. **Mineração de Dados e Interpretação**: aplicação comparativa do **K-Means** como *baseline* particional, do **DBSCAN** para isolamento de anomalias e possíveis fraudes por densidade, e do **GMM (*Gaussian Mixture Models*)** para segmentação probabilística (*soft clustering*) dos perfis de consumo.

## 1. Importação das Bibliotecas

Importamos as bibliotecas de análise de dados (`pandas`, `numpy`), visualização (`matplotlib`, `seaborn`), pré-processamento e agrupamento (`scikit-learn`), além do módulo `util.py`, que armazena apenas as funções utilitárias de reuso recorrente (carregamento, diagnóstico de estrutura, plotagem padronizada de histograma com boxplot e cálculo consolidado de métricas de validação de clusters).

In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import util

util.configure_plot_style()

## 2. Etapa de Seleção (KDD): Importação e Diagnóstico Inicial da Base

Carregamos a base `base04_cartao_alta_dimensao.csv` e verificamos suas dimensões, a quantidade de colunas, os tipos de variáveis e a presença de dados nulos.

### O que escolhemos e por que não escolhemos outra abordagem
- **Ausência de Imputação de Dados Nulos**: Todas as 37 colunas possuem exatamente 5.000 registros válidos (0 valores ausentes). Por essa razão, não aplicamos técnicas de imputação por média, mediana ou *KNN Imputer*, evitando alterar a distribuição real dos dados.
- **Remoção do Atributo `id_cliente` na Modelagem**: A coluna `id_cliente` (`C000001` a `C005000`) é apenas uma chave identificadora única de banco de dados. Mantê-la ou transformá-la em número criaria uma relação artificial baseada na ordem de cadastro.

In [2]:
raw_df = util.load_data("../data/raw/base04_cartao_alta_dimensao.csv")
util.show_info(raw_df)

Dimensoes da Base: 5000 linhas x 37 colunas
Total de Valores Ausentes (Nulos): 0

Contagem de Tipos de Variaveis:
float64    19
int64      15
object      3

Valores Nulos por Coluna:
id_cliente                          0
limite_total_credito                0
limite_utilizado_pct                0
gasto_medio_mensal_3m               0
gasto_medio_mensal_12m              0
gasto_maximo_unico                  0
qtd_transacoes_30d                  0
qtd_transacoes_90d                  0
qtd_estornos_12m                    0
valor_estornos_12m                  0
taxa_estorno_pct                    0
qtd_atrasos_12m                     0
dias_desde_ultimo_atraso            0
pct_pagamento_minimo                0
pct_pagamento_total_fatura          0
qtd_parcelamentos_12m               0
pct_compras_parceladas              0
pct_compras_online                  0
pct_compras_internacionais          0
pct_compras_madrugada               0
pct_compras_fim_de_semana           0
qtd_lojistas_distin